<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">MODULE 13: STORAGE AND LIFECYCLE MANAGEMENT</div><div style="color:#17212b;font-size:30px;font-weight:750">Storage and lifecycle management</div><p style="color:#475569;line-height:1.7">Run in order against a dedicated Level 3 course database. Results are rendered as tables and all examples are scoped to this module.</p></div>

## Boundary

Do not grant access to, truncate, or alter a shared production object from this lab. Review every object name before executing a write.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP TABLE IF EXISTS ops_orders_l3")
lab.execute("""
CREATE TABLE ops_orders_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01"),
    PARTITION p202503 VALUES LESS THAN ("2025-04-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("ops_orders_l3", ["order_date", "order_id", "status", "amount"], [("2025-01-15", 130001, "PAID", 100), ("2025-02-15", 130002, "PAID", 200), ("2025-03-15", 130003, "SHIPPED", 300)])
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="Isolated lifecycle table")

In [ ]:
lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="Partition and row-count evidence")
lab.sql("SHOW TABLETS FROM ops_orders_l3", title="Tablet layout evidence")
lab.sql("SHOW CREATE TABLE ops_orders_l3", title="Retention and distribution contract")

In [ ]:
lab.execute("TRUNCATE TABLE ops_orders_l3 PARTITION (p202501)")
lab.sql("SELECT * FROM ops_orders_l3 ORDER BY order_date, order_id", title="Remaining data after January retention")
lab.sql("SHOW PARTITIONS FROM ops_orders_l3 ORDER BY PartitionName", title="Post-maintenance metadata", final=True)

## Takeaway

Operational correctness includes scope, evidence, reversibility, and ownership. Record what changed and how the result was checked before declaring the exercise complete.